# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR² dataset package using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.github.io/croissant/), available via the following URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

***
**About this dataset:**
> This dataset contains ordered logistic regression outputs including log likelihood values across iterations, coefficients, standard errors, and p-values for variables affecting household adoption of indigenous and modern knowledge in rangeland management interventions. The data covers socio-demographic characteristics, knowledge management processes, and intervention outcomes among pastoral households in Samburu, Isiolo, and Marsabit counties, Northern Kenya.

In [ ]:
# Install mlcroissant if it is not already installed
!pip install --quiet mlcroissant

## 1. Data Loading
Let's load the dataset metadata and explore its contents using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata for overview
metadata = dataset.metadata
print(f"Name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Let's examine the available record sets (tables), their fields (columns), and their `@id` values. Referencing by `@id` ensures precise access to the underlying schema entities.

> If the dataset contains multiple record sets, all will be listed.

In [ ]:
# List all record sets and their @id, name, and available fields
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    # If there are no record sets in the metadata, try getting them from the dataset object
    record_sets = list(dataset.record_sets())

print('Discovered record sets:')
record_set_overview = {}
for rs in record_sets:
    # Some mlcroissant APIs return @id, some objects, keep both
    rs_obj = dataset.record_set(rs) if isinstance(rs, str) else rs
    rs_id = rs if isinstance(rs, str) else getattr(rs, '@id', getattr(rs, 'id', None))
    rs_name = getattr(rs_obj, 'name', None)
    # Get list of fields (by @id and name if possible)
    if hasattr(rs_obj, 'fields'):
        fields = rs_obj.fields
    else:
        fields = getattr(rs_obj, 'field', [])
    field_ids = []
    field_labels = []
    for f in fields:
        # f may be an @id string or an object
        if isinstance(f, str):
            try:
                f_obj = dataset.field(f) # Try loading the field object
            except Exception:
                f_obj = None
            field_ids.append(f)
            field_labels.append(getattr(f_obj, 'name', f) if f_obj else f)
        else:
            field_ids.append(getattr(f, '@id', getattr(f, 'id', None)))
            field_labels.append(getattr(f, 'name', ''))
    record_set_overview[rs_id] = {'name': rs_name, 'fields': field_ids}
    field_list = ', '.join([f"{fid}" for fid in field_ids])
    print(f"Record Set: {rs_name or rs_id}\n  @id: {rs_id}\n  Fields (@id): {field_list}\n")

if not record_set_overview:
    print("No record sets discovered in the metadata. Check dataset schema.")

## 3. Data Extraction
We'll now extract tabular data from each discovered record set (using its `@id`). All columns will be referenced by their `@id` for clarity and reproducibility.

Data is loaded into a dictionary of DataFrames for downstream processing.

In [ ]:
# Prepare to extract all record sets into pandas DataFrames
dataframes = {}
# If no record sets, fall back to printing a message
if not record_set_overview:
    print("No record sets available to extract.")
else:
    for record_set_id in record_set_overview:
        try:
            records = list(dataset.records(record_set=record_set_id))
            if len(records) == 0:
                print(f"No records found in record set {record_set_id}.")
                continue
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Extracted {len(df)} records from record set '{record_set_id}'. Columns (by @id):")
            print(list(df.columns))
        except Exception as ex:
            print(f"Could not extract records from record set '{record_set_id}': {ex}")

# For demonstration, pick the first record set for display (replace this @id as needed)
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print("\nPreview of first few records from record set:")
    display(dataframes[first_record_set_id].head())
else:
    print("No DataFrames created. Cannot preview data.")

## 4. Exploratory Data Analysis (EDA)
Let's apply some typical data processing: select a numeric field (by `@id`), filter records (e.g., by value threshold), normalize, and group by a categorical field using their Croissant `@id` references.

In [ ]:
# Example: Pick a record set for EDA
# Use the first loaded record set, or replace with a specific @id
if dataframes:
    record_set_id = first_record_set_id
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")
    # List numeric candidate fields by checking dtypes
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric fields (@id): {numeric_candidates}")
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # For demo, pick the first numeric field
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"\nFiltered records with {numeric_field} > {threshold:.2f} (showing up to 5 rows):")
        display(filtered_df.head(5))

        # Normalizing the selected numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(5))

        # Try grouping by a categorical field (if exists)
        categorical_candidates = df.select_dtypes(include=['object']).columns.tolist()
        group_field = None
        if categorical_candidates:
            # Pick the first non-numeric string field as group
            group_field = categorical_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            print(f"\nGrouped data by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No categorical group_field found in DataFrame.")
    else:
        print("No numeric fields detected – cannot demonstrate numeric EDA.")
else:
    print("No extracted data available for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and if a grouping field is identified, plot the group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if numeric field available from previous analysis
if dataframes and 'numeric_field' in locals() and numeric_field in df:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    if 'group_field' in locals() and group_field and group_field in df:
        plt.figure(figsize=(10, 6))
        sns.barplot(y=grouped_df.index, x=grouped_df[f"mean_{numeric_field}"], orient='h')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.show()
else:
    print("No visualization: Numeric field not available or DataFrame missing.")

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load Croissant schema-based datasets using `mlcroissant`
- Discover record sets and fields using their `@id` references
- Extract data into pandas DataFrames, referencing schema entities by `@id`
- Perform basic EDA including filtering, normalization, and grouping
- Visualize data distributions and group-level statistics

**Next steps:**
- Explore additional record sets (if available)
- Apply advanced analytics or modeling as needed
- Consult the dataset's documentation and Croissant schema for further details

> **Reminder:** Always refer to fields, columns, record sets, and other schema entities by their `@id` in code for full reproducibility and future-proof workflows.